# GNN Laundering Inference by Account

This notebook loads the saved **GraphSAGE A+B (Synergy)** model from `run_1_GraphSAGE_A+B_(Synergy).pkl` and provides an interface to run the model for a given **account ID** and get a **laundering** prediction (risk score and binary flag).

**Requirements:**
- The model file: `backend/model/run_1_GraphSAGE_A+B_(Synergy).pkl`
- Processed graph and features from the original pipeline (same format as `gnn_fraud_detection_iis_project_amin.ipynb`). Set `PROCESSED_DATA_DIR` below to the path of your `processed_data` folder (containing `metadata/`, `features/`, `feature_manifest.json`). If you only have the `.pkl`, run the original notebook once to generate the processed data, then point this path to it.

In [1]:
# ============================================
# Paths & Imports (reused from gnn_fraud_detection_iis_project_amin.ipynb)
# ============================================
import os
import json
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv

# --- Paths ---
# Run this notebook from GenAI-Genesis folder, or set PROJECT_ROOT to the repo root
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "backend" / "model").exists():
    pass  # already at repo root
else:
    PROJECT_ROOT = PROJECT_ROOT / "GenAI-Genesis" if (PROJECT_ROOT / "GenAI-Genesis").exists() else PROJECT_ROOT

MODEL_PATH = PROJECT_ROOT / "backend" / "model" / "run_1_GraphSAGE_A+B_(Synergy).pkl"
# Point this to your processed_data folder (from running the original notebook)
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"
OUTPUT_DIR = Path(PROCESSED_DATA_DIR)
FEATURE_DIR = OUTPUT_DIR / "features"
META_DIR = OUTPUT_DIR / "metadata"

# Default risk threshold for binary "is_laundering" (tune as needed)
DEFAULT_RISK_THRESHOLD = 0.5

print(f"Model path: {MODEL_PATH}")
print(f"Processed data dir: {PROCESSED_DATA_DIR}")
print(f"Model exists: {MODEL_PATH.exists()}")
print(f"Processed data exists: {OUTPUT_DIR.exists()}")

c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model path: c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis\backend\model\run_1_GraphSAGE_A+B_(Synergy).pkl
Processed data dir: c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis\processed_data
Model exists: True
Processed data exists: False


## Model architecture (same as in gnn_fraud_detection_iis_project_amin.ipynb)

Copy-pasted: `GraphSAGE_AML`, `GCN_AML`, `GAT_AML`, and `build_model` so the loaded state_dict matches.

In [2]:
# ============================================
# GNN Model Architectures (The Model Zoo) - same as original notebook
# ============================================

class GCN_AML(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=3, dropout=0.3, num_classes=2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.cls = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.cls(x)
        return F.log_softmax(x, dim=1)


class GraphSAGE_AML(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=3, dropout=0.3, aggr="mean", num_classes=2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(SAGEConv(input_dim, hidden_dim, aggr=aggr))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim, aggr=aggr))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.cls = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x_prev = x
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            if x.shape == x_prev.shape:
                x = x + x_prev
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.cls(x)
        return F.log_softmax(x, dim=1)


class GAT_AML(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, heads=2, dropout=0.3, num_classes=2, gat_concat=False):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(input_dim, hidden_dim, heads=heads, concat=gat_concat, dropout=dropout))
        in_dim = (hidden_dim * heads) if gat_concat else hidden_dim
        for _ in range(max(num_layers - 2, 0)):
            self.convs.append(GATConv(in_dim, hidden_dim, heads=heads, concat=gat_concat, dropout=dropout))
            in_dim = (hidden_dim * heads) if gat_concat else hidden_dim
        self.convs.append(GATConv(in_dim, hidden_dim, heads=1, concat=False, dropout=dropout))
        self.cls = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        x = F.elu(x)
        x = self.cls(x)
        return F.log_softmax(x, dim=1)


def build_model(model_name, input_dim, cfg):
    """Factory function for model instantiation (same as original notebook)."""
    if model_name == "GCN":
        return GCN_AML(input_dim=input_dim, hidden_dim=cfg["hidden_dim"], num_layers=cfg["num_layers"], dropout=cfg["dropout"])
    elif model_name == "GraphSAGE":
        return GraphSAGE_AML(input_dim=input_dim, hidden_dim=cfg["hidden_dim"], num_layers=cfg["num_layers"], dropout=cfg["dropout"], aggr=cfg.get("aggr", "mean"))
    elif model_name == "GAT":
        return GAT_AML(input_dim=input_dim, hidden_dim=cfg["hidden_dim"], num_layers=cfg["num_layers"], heads=cfg["heads"], dropout=cfg["dropout"], gat_concat=cfg.get("gat_concat", False))
    raise ValueError(f"Unknown model architecture: {model_name}")

print("Model zoo (GCN, GraphSAGE, GAT) and build_model ready.")

Model zoo (GCN, GraphSAGE, GAT) and build_model ready.


In [3]:
# ============================================
# Feature loading (same as original notebook)
# ============================================

def load_feature_block(block_key: str, split: str = "train", output_dir: Path = None):
    """Helper to load a specific feature block from the manifest registry. Resolves paths relative to OUTPUT_DIR if needed."""
    out = output_dir or OUTPUT_DIR
    with open(out / "feature_manifest.json", "r") as f:
        man = json.load(f)
    path = Path(man["feature_blocks"][block_key]["paths"][split])
    if not path.exists():
        path = FEATURE_DIR / path.name
    return torch.load(path, weights_only=False)


def combine_feature_blocks(block_keys, split: str = "train", output_dir: Path = None):
    """Concatenates multiple feature blocks into a single high-dimensional tensor."""
    blocks = [load_feature_block(k, split=split, output_dir=output_dir) for k in block_keys]
    X = torch.cat(blocks, dim=1)
    return X

# A+B (Synergy) = behavioral_A + random_walk_B (same as run_1 model)
FEATURE_BLOCKS_A_B = ["behavioral_A", "random_walk_B"]
print("Feature loaders ready. A+B blocks:", FEATURE_BLOCKS_A_B)

Feature loaders ready. A+B blocks: ['behavioral_A', 'random_walk_B']


In [4]:
# ============================================
# Load saved model from run_1_GraphSAGE_A+B_(Synergy).pkl
# ============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model_name = ckpt["model_name"]
input_dim = int(ckpt["input_dim"])
cfg = ckpt["config"]
feature_set = ckpt.get("feature_set", "A+B (Synergy)")

model = build_model(model_name, input_dim=input_dim, cfg=cfg)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(device)
model.eval()

print(f"Loaded: {model_name} | feature_set={feature_set} | input_dim={input_dim}")

Loaded: GraphSAGE | feature_set=A+B (Synergy) | input_dim=58


In [ ]:
# ============================================
# Load graph data and account maps (required for account -> node lookup)
# ============================================
if not OUTPUT_DIR.exists():
    print("WARNING: PROCESSED_DATA_DIR not found. Set PROCESSED_DATA_DIR to the folder containing metadata/ and features/ (from the original notebook).")
    base_data = None
    account_to_id = None
    id_to_account = None
    X_full = None
    edge_index_full = None
else:
    base_data = torch.load(META_DIR / "base_graph_data.pt", map_location=device, weights_only=False)
    edge_index_full = base_data.edge_index.to(device)
    with open(META_DIR / "account_maps.pkl", "rb") as f:
        maps = pickle.load(f)
    account_to_id = maps["account_to_id"]
    id_to_account = maps["id_to_account"]
    # A+B features for the test split (use same split for consistency; all nodes have features)
    X_full = combine_feature_blocks(FEATURE_BLOCKS_A_B, split="test").to(device)
    if X_full.shape[1] != input_dim:
        raise ValueError(f"Feature dim {X_full.shape[1]} != model input_dim {input_dim}. Use A+B blocks only.")
    print(f"Graph: {base_data.num_nodes:,} nodes, {base_data.num_edges:,} edges")
    print(f"Accounts in map: {len(account_to_id):,}")

In [ ]:
# ============================================
# Run model on full graph and cache node-level risk scores
# ============================================
node_risk_scores = None  # shape (num_nodes,) probability of class 1 (laundering)

if X_full is not None and edge_index_full is not None:
    with torch.no_grad():
        log_probs = model(X_full, edge_index_full)
    # Class 1 = laundering
    node_risk_scores = log_probs.exp()[:, 1].cpu().numpy()
    print("Inference done. Node risk scores cached.")
else:
    print("Skipping inference (no graph/features loaded).")

In [ ]:
# ============================================
# Interface: get laundering prediction for an account
# ============================================

def get_laundering_prediction(account_id, risk_threshold=None):
    """
    Given an account ID (int or str), return whether the model flags it as laundering and the risk score.

    Returns:
        dict with keys: is_laundering (bool), risk_score (float), node_id (int or None), message (str)
    """
    if risk_threshold is None:
        risk_threshold = DEFAULT_RISK_THRESHOLD
    if node_risk_scores is None or account_to_id is None:
        return {
            "is_laundering": None,
            "risk_score": None,
            "node_id": None,
            "message": "Graph/account maps not loaded. Set PROCESSED_DATA_DIR and run the load cells.",
        }
    # Normalize account_id (map may use int or str)
    key = account_id if account_id in account_to_id else None
    if key is None and isinstance(account_id, str) and account_id.isdigit():
        key = int(account_id)
    if key is None and isinstance(account_id, int):
        key = str(account_id)
    if key not in account_to_id:
        return {
            "is_laundering": None,
            "risk_score": None,
            "node_id": None,
            "message": f"Account '{account_id}' not found in graph.",
        }
    node_id = account_to_id[key]
    risk = float(node_risk_scores[node_id])
    return {
        "is_laundering": risk >= risk_threshold,
        "risk_score": risk,
        "node_id": int(node_id),
        "message": "OK",
    }


# Example usage (run after the above cells):
# result = get_laundering_prediction(12345)
# print(result)  # e.g. {"is_laundering": False, "risk_score": 0.12, "node_id": 42, "message": "OK"}

## Try it: run prediction for an account

Call `get_laundering_prediction(account_id)` with an account ID that exists in the graph. Below we sample a few accounts if data is loaded.

In [ ]:
# Example: predict for one or more accounts
if id_to_account is not None and len(id_to_account) > 0:
    # Sample a few account IDs from the graph
    sample_ids = list(id_to_account.values())[:5]
    for acc in sample_ids:
        r = get_laundering_prediction(acc)
        print(f"Account {acc} -> is_laundering={r['is_laundering']}, risk_score={r['risk_score']:.4f} ({r['message']})")
else:
    # No data loaded: show how to call the API
    print("No account map loaded. Example: result = get_laundering_prediction(<your_account_id>)")
    print("Returns: is_laundering (bool), risk_score (float), node_id, message")